# WeatherNext Cyclones Mini: control vs. K&C bogus for Typhoon Mawar (WP02 2023)

Runs one member of DeepMind's WeatherNext 2 / WeatherNext Cyclones family
(the FGN architecture) twice from the same ERA5 analysis at
**2023-05-26 06Z**, about a day before Mawar's peak (`INIT = "peak"`,
the default), or at **2023-05-20 12Z**, when the best track first has it as a
tropical storm (35 kt, 996 hPa; `INIT = "ts"`):

1. **control**: the unaltered ERA5 state;
2. **bogus**: the ERA5 state with a Kwon & Cheong (2009) vortex matched to the
   JTWC best-track intensity.

The model's native step is 6 h, so both runs go 72 h in 12 steps. At each step
the storm is located by `tcinit.rollout_tracking.track_rollout`, which centres
each box on the previous step's position, and the two runs are compared
against the best track.

**Run on:** a cee-compute GPU node in `jax-venv` (see `envs/README.md`). The
config ships with the `weathernext` package; the weights and ERA5 are read
anonymously from Google Cloud Storage, so the node needs outbound network
access (proxy set before Jupyter starts).

**Checkpoint: `WeatherNextCyclones_Mini_<2023`.** The full 0.25° WN2 and
WN Cyclones checkpoints need H100 memory on GPU; the node has A100 80GB, so this
uses a Mini (1°, 13 pressure levels). Of the two Minis, `<2024` was trained on
data through 2023 and so has seen Mawar; `<2023` was trained through 2022, so
this forecast is out of sample. The WeatherNext demo initialises the Minis from
HRES; here ERA5 is used, to match the GraphCast notebook.

**GPU attention.** The checkpoints are configured for TPU splash attention.
As the WeatherNext README and `wn2_demo.ipynb` say, on GPU the transformer must
be switched to `attention_type="triblockdiag_mha"` (section 6).

**Time ordering.** WeatherNext inputs carry two frames on a `time` axis of
lead times `[-6 h, 0 h]`, oldest first, so the current state is last
(`time_idx=-1`). Section 3 ends with a check of this against the best
track, before anything runs.

## 1. Configuration

In [ ]:
import os
from datetime import datetime, timedelta
from pathlib import Path

# Pick one A100. Must be set before jax is imported.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# JAX grabs 75% of GPU memory on first use. Uncomment on a shared node.
# os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

PROJECT = Path("/projects/aurora")
BEST_TRACK_FILE = PROJECT / "best-tracks/westpac/bwp2023/bwp022023.dat"
ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
WN_BUCKET = "dm_graphcast"
MODEL_NAME = "WeatherNextCyclones_Mini"
SPLIT = "2023"  # trained on data through 2022; Mawar is out of sample
CONFIG_NAME = f"weathernext2/configs/{MODEL_NAME}"  # inside the weathernext package
PARAMS_FILE = f"weathernext2/params/{MODEL_NAME}_<{SPLIT}.npz"
COARSEN = 4  # 0.25° ERA5 -> the Mini's 1° grid
MODEL_CACHE = Path(f"/fgs/{os.environ['USER']}/causaltc/model-cache/weathernext")
JAX_CACHE = Path(f"/fgs/{os.environ['USER']}/causaltc/jax_cache")

# Initial time. "peak": 2023-05-26 06Z, about a day before Mawar's peak.
# "ts": 2023-05-20 12Z, the first best-track fix at tropical-storm
# strength (35 kt, 996 hPa, near 5.6 N 149.4 E).
# The model also reads the t - 6 h state; for "ts" that is 2023-05-20 06Z,
# a 30 kt tropical depression in the best track.
# "ts" writes to its own OUT_DIR (suffix "_ts"), so the two cases don't
# overwrite each other's plots or saved rollouts.
INIT = "peak"
START_TIMES = {"peak": datetime(2023, 5, 26, 6), "ts": datetime(2023, 5, 20, 12)}
START_TIME = START_TIMES[INIT]
INIT_TAG = "" if INIT == "peak" else f"_{INIT}"
OUT_DIR = Path(f"/fgs/{os.environ['USER']}/causaltc/demo_out/wn2_mawar{INIT_TAG}")

STEP = timedelta(hours=6)  # the model's native step
N_STEPS = 12  # 72 h
BOX_HALF_DEG = 10.0

# Blend the K&C vortex's omega into `vertical_velocity` (section 5). The bogus
# rollout and its plots get an "_omega" suffix, so runs with and without the
# blend are kept side by side in OUT_DIR.
BLEND_OMEGA = True
OMEGA_TAG = "_omega" if BLEND_OMEGA else ""
BOGUS_LABEL = f"bogus{OMEGA_TAG}"

# Reuse the tracked rollouts in OUT_DIR from an earlier run instead of
# running the forecasts again. Set True to force a rerun, including after
# changing SPLIT (the saved rollouts don't record it).
RERUN = False

if not BEST_TRACK_FILE.exists():
    raise FileNotFoundError(BEST_TRACK_FILE)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
JAX_CACHE.mkdir(parents=True, exist_ok=True)
print(f"output -> {OUT_DIR}")


def saved_rollout(label):
    # Directory written by an earlier `<rollout>.to_netcdf(OUT_DIR / label)`, or None.
    hits = sorted((OUT_DIR / label).glob(f"*/{START_TIME:%Y%m%d%H}_steps*/index.json"))
    return hits[-1].parent if hits else None


REUSE = not RERUN and all(saved_rollout(k) for k in ("control", BOGUS_LABEL))
print("reusing saved rollouts" if REUSE else "will run WN Cyclones Mini")

In [ ]:
import dataclasses
import functools
import time
from contextlib import contextmanager
from importlib import metadata

import gcsfs
import haiku as hk
import jax
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from weathernext.utils import checkpoint, data_utils, fiddle_config_io, rollout
from weathernext.weathernext2 import fgn

from tcinit import BestTrack
from tcinit.backends import weathernext2_backend as wn_backend
from tcinit.constants import HPA_TO_PA
from tcinit.naming import WEATHERNEXT_VAR_MAP
from tcinit.network import use_env_proxy_for_aiohttp
from tcinit.plotting import plot_snapshot, plot_snapshot_diagnostics
from tcinit.rollout_snapshots import (
    RolloutSnapshots,
    plot_best_track_comparison,
    plot_rollout_comparison,
    plot_rollout_snapshots,
)
from tcinit.rollout_tracking import build_bogus, snapshot_from_box, track_rollout

# Persistent compilation cache, shared with the other jax notebooks: the
# first run compiles the model step, later runs load it from disk.
# Changing jax, weathernext or XLA_FLAGS invalidates the entries.
jax.config.update("jax_compilation_cache_dir", str(JAX_CACHE))
jax.config.update("jax_persistent_cache_min_compile_time_secs", 10)

# gcsfs (ERA5 and the weights on gs://) uses aiohttp, which ignores
# HTTPS_PROXY unless told otherwise; without this the Sandia web filter
# blocks the downloads.
use_env_proxy_for_aiohttp()

for dist in ("jax", "dm-haiku", "weathernext", "xarray"):
    print(f"{dist:12s}", metadata.version(dist))
print("devices:", jax.devices())

## 2. Best track

In [ ]:
bt = BestTrack(BEST_TRACK_FILE)
bt_center = bt.get_location(START_TIME)
print(
    f"{bt.ID} at {START_TIME:%Y-%m-%d %HZ}: "
    f"({bt_center[0]:.1f}N, {bt_center[1]:.1f}E), "
    f"{bt.get_windspeed(START_TIME):.0f} m/s, "
    f"{bt.get_central_pressure(START_TIME) / HPA_TO_PA:.0f} hPa"
)

## 3. Initial state from ERA5

The model config is read first, because its `task` lists the input and target
variables and pressure levels; the network itself is built in section 6. The
weights come from the `dm_graphcast` bucket and are cached under
`MODEL_CACHE`. The normalization statistics are part of the config.

The model takes two input frames, `[t − 6 h, t]`. The fields come from the
ARCO ERA5 store, subsampled to 1° (every 4th point; the grid is 181 × 360 with
both poles, as in the WeatherNext 1° example data), renamed to WeatherNext's
`lat`/`lon` and flipped to ascending latitude. Sea-surface temperature is NaN
over land, as in training; the config's NaN-cleaning wrapper fills it inside
the model.

The inputs, targets template and forcings are then built the way the
WeatherNext demo builds them: one dataset with the two input frames followed
by `N_STEPS` target frames, split by `data_utils.extract_inputs_targets_forcings`.
That function also computes the year/day-progress forcings at every frame from
the `datetime` coordinate. The target frames are lazy all-NaN dask arrays.
Variables that are targets only (`total_precipitation_6hr` and the gridded
`cyclone_*` heads) appear in the dataset only as NaN placeholders with dims
`(batch, time, lat, lon)`, matching the WeatherNext example files.

In [ ]:
GCS = gcsfs.GCSFileSystem(token="anon")


def cached(path):
    # Local copy of gs://dm_graphcast/<path>, downloaded on first use.
    local = MODEL_CACHE / path
    if not local.exists():
        local.parent.mkdir(parents=True, exist_ok=True)
        GCS.get(f"{WN_BUCKET}/{path}", str(local))
    return local


config = fiddle_config_io.get_fiddle_config_by_name(CONFIG_NAME)
task_config = config.task
with open(cached(PARAMS_FILE), "rb") as f:
    ckpt = checkpoint.load(f, fgn.CheckPoint)
print(ckpt.description)
print("levels:", task_config.pressure_levels, " input duration:", task_config.input_duration)

In [ ]:
STEP_H = int(STEP / timedelta(hours=1))
N_FRAMES = 2 + N_STEPS  # two input frames, then the target lead times
DATETIMES = [START_TIME + (k - 1) * STEP for k in range(N_FRAMES)]
HISTORY = tuple(DATETIMES[:2])  # oldest first, current last
LEVELS = list(task_config.pressure_levels)
STATIC = ("geopotential_at_surface", "land_sea_mask")
# Time-dependent inputs read from ERA5; the forcings are computed below.
DYNAMIC = [
    v
    for v in task_config.input_variables
    if v not in task_config.forcing_variables and v not in STATIC
]
TARGET_ONLY = [v for v in task_config.target_variables if v not in DYNAMIC]
print("from ERA5:", DYNAMIC)
print("target-only placeholders:", TARGET_ONLY)

era5 = xr.open_zarr(ERA5_ZARR, chunks=None, storage_options=dict(token="anon"))


def to_wn_grid(ds):
    # ARCO 0.25° (latitude 90 -> -90, longitude) -> WeatherNext 1° (lat -90 -> 90, lon).
    ds = ds.isel(latitude=slice(None, None, COARSEN), longitude=slice(None, None, COARSEN))
    ds = ds.rename(latitude="lat", longitude="lon").sortby("lat")
    ds = ds.assign_coords(lat=ds.lat.astype(np.float32), lon=ds.lon.astype(np.float32))
    if "level" in ds.coords:
        ds = ds.assign_coords(level=ds.level.astype(np.int32))
    return ds


def era5_field(name, t):
    da = era5[name].sel(time=np.datetime64(t))
    if "level" in da.dims:
        da = da.sel(level=LEVELS)
    return da.reset_coords(drop=True).astype(np.float32)


LEAD = (np.arange(N_FRAMES) * STEP_H).astype("timedelta64[h]").astype("timedelta64[ns]")
frames = xr.concat(
    [
        to_wn_grid(xr.Dataset({v: era5_field(v, t) for v in DYNAMIC})).load().expand_dims(
            time=[LEAD[k]]
        )
        for k, t in enumerate(HISTORY)
    ],
    dim="time",
)
statics = to_wn_grid(
    xr.Dataset(
        {v: era5[v].sel(time=np.datetime64(START_TIME)).reset_coords(drop=True) for v in STATIC}
    )
).load()

# Lazy NaN frames for the target lead times.
pad = xr.full_like(frames.isel(time=[-1]).chunk(), np.nan)
pads = xr.concat([pad.assign_coords(time=[LEAD[k]]) for k in range(2, N_FRAMES)], dim="time")
dyn = xr.concat([frames, pads], dim="time")
for v in TARGET_ONLY:
    dyn[v] = xr.full_like(dyn["2m_temperature"], np.nan)
example = dyn.expand_dims("batch").assign({v: statics[v] for v in STATIC})
example = example.assign_coords(
    datetime=(("batch", "time"), np.array(DATETIMES, dtype="datetime64[ns]")[None])
)

inputs, targets_template, forcings = data_utils.extract_inputs_targets_forcings(
    example,
    target_lead_times=slice(f"{STEP_H}h", f"{N_STEPS * STEP_H}h"),
    **dataclasses.asdict(task_config),
)
inputs = inputs.compute()
forcings = forcings.compute()
assert (inputs.sizes["lat"], inputs.sizes["lon"]) == (181, 360), dict(inputs.sizes)

print("inputs:   ", dict(inputs.sizes), inputs.time.values.astype("timedelta64[h]"))
print("targets:  ", dict(targets_template.sizes))
print("forcings: ", dict(forcings.sizes), list(forcings.data_vars))

In [ ]:
# Ordering check: slot i should hold the storm at HISTORY[i].
for i, t in enumerate(HISTORY):
    center = bt.get_location(t)
    box = wn_backend.extract_snapshot_dataset(
        inputs, box_center=center, box_size=BOX_HALF_DEG, time_idx=i
    )
    s = snapshot_from_box(
        box, valid_time=t, search_center=center, var_map=WEATHERNEXT_VAR_MAP
    )
    print(
        f"slot {i} ({t:%d %HZ}): model storm at ({s.loc[0]:.2f}, {s.loc[1]:.2f}), "
        f"{s.mslp_min / HPA_TO_PA:.1f} hPa | best track at ({center[0]:.2f}, {center[1]:.2f})"
    )

## 4. Control initial state

`time_idx=-1` is the current slot (lead time 0 h). This is also
`wn_backend.extract_snapshot_dataset`'s default; it is passed explicitly here
because the same `extract_box` reads the forecast chunks, which have a single
time slot. The box keeps WeatherNext's long-form variable names
(`mean_sea_level_pressure`, `temperature`, ...); `var_map=WEATHERNEXT_VAR_MAP`
tells `snapshot_from_box`, `build_bogus`, `track_rollout` and `plot_snapshot`
how to read them.

At 1° a `BOX_HALF_DEG = 10` box is only 21 × 21 points, so both the detected
storm and the bogus vortex are coarse compared with the 0.25° notebooks.

In [ ]:
def extract_box(state, center):
    return wn_backend.extract_snapshot_dataset(
        state, box_center=center, box_size=BOX_HALF_DEG, time_idx=-1
    )


box_ctrl0 = extract_box(inputs, bt_center)
snap_ctrl0 = snapshot_from_box(
    box_ctrl0, valid_time=START_TIME, search_center=bt_center, var_map=WEATHERNEXT_VAR_MAP
)
plot_snapshot(
    snap_ctrl0,
    box_ctrl0,
    var_map=WEATHERNEXT_VAR_MAP,
    title=f"Mawar, ERA5 1° {START_TIME:%Y-%m-%d %HZ} (control initial state)",
    filename=OUT_DIR / "snapshot_control_t0.png",
)
plt.show()

# Gradient-wind balance of the same state: V_t vs. the gradient wind of its
# own geopotential (tcinit.plotting.plot_snapshot_diagnostics).
plot_snapshot_diagnostics(
    snap_ctrl0,
    title=(
        f"Mawar, ERA5 1° {START_TIME:%Y-%m-%d %HZ} (control initial state)"
        " — gradient-wind balance"
    ),
    filename=OUT_DIR / "balance_control_t0.png",
)
plt.show()

## 5. Bogus initial state

`build_bogus` detects the storm, builds the K&C vortex from the best-track
intensity, and applies it to the box with the backend's `apply`. It runs once
per input frame: slot 0 (t − 6 h) gets a vortex built from the best track at
that time, centred on the storm's position then, and slot 1 (t) one from
`START_TIME`. Each is written into its own frame only (`time_slots=(i,)`), so
between the two frames the bogus storm moves and changes intensity as the best
track does. Writing one vortex into both frames (the `write_back` default,
`time_slots=None`) gives the model a storm that is not moving. In the
GraphCast demo, done that way, the bogus storm moved 0.25° in the first 6 h
against the control's 1.25°.

`wn_backend.write_back` (the WN2 adapter over the WN-Graph one) returns a deep
copy of the inputs with the box pasted in. It writes `msl`, `2t`, `10u`, `10v`
and `t`, `u`, `v`, `q`, `z` on every level the box shares with the inputs,
across all batch entries. The WN2 adapter also always skips the 100 m winds
(the Mini has none, so that is a no-op here). Fields the vortex does not model
(`sea_surface_temperature`, the forcings and statics) are left unchanged.

With `BLEND_OMEGA` (section 1), `vertical_velocity` is blended too, with the
same weight as `u` and `v`, toward the omega that K&C's radial inflow and
outflow imply through continuity (`bv.omega_bogus_pa_s`). Without it, the
bogus convergence sits next to the analysis omega, which the model reads as
an input. The omega is computed from the analytic radial wind, not from the
gridded winds: on the grid, the tangential wind's differencing error near the
radius of maximum wind would add O(10) Pa/s of spurious omega.
`wn_backend.omega_diagnostics` prints, per frame:

- `omega_increment`: the change to omega in the blend region (Pa/s).
- `surface_residual`, `continuity_after`: how far the gridded winds are from
  column mass balance and from discrete continuity with the blended omega.
  Both are dominated by the grid's tangential-wind error.
- `hydrostatic_before` / `hydrostatic_after`: layer virtual-temperature
  error (K) implied by the geopotential. Most of the change comes from the
  humidity blend, which raises T_v without adjusting `z`, not from omega.
- `nonhydrostatic_ratio`: |Dw/Dt| / g of the blended omega; ≪ 1 means the
  hydrostatic continuity used to derive it holds.

If the model's inputs have no `vertical_velocity`, the blend is skipped and
the cell says so.

In [ ]:
# One vortex per input frame, from the best track at that frame's time.
blend_omega = BLEND_OMEGA and "vertical_velocity" in inputs
if BLEND_OMEGA and not blend_omega:
    print("inputs have no vertical_velocity; skipping the omega blend")
inputs_bogus = inputs
for i, t in enumerate(HISTORY):
    center = bt.get_location(t)
    box = wn_backend.extract_snapshot_dataset(
        inputs,
        box_center=center,
        box_size=BOX_HALF_DEG,
        time_idx=i,
        include_omega=blend_omega,
    )
    _, bv, box_bogus = build_bogus(
        box,
        bt,
        t,
        search_center=center,
        var_map=WEATHERNEXT_VAR_MAP,
        apply=functools.partial(wn_backend.apply, blend_omega=blend_omega),
    )
    if blend_omega:
        diag = wn_backend.omega_diagnostics(box, box_bogus, bv)
        print(
            f"  slot {i} omega: K&C min {bv.omega_bogus_pa_s.min():.2f} Pa/s | "
            + ", ".join(f"{k} max {v['max_abs']:.3g}" for k, v in diag.items())
        )
    inputs_bogus = wn_backend.write_back(
        inputs_bogus,
        box_bogus,
        box_lats=box.latitude.values,
        box_lons=box.longitude.values,
        time_slots=(i,),
    )
    s = snapshot_from_box(
        wn_backend.extract_snapshot_dataset(
            inputs_bogus, box_center=center, box_size=BOX_HALF_DEG, time_idx=i
        ),
        valid_time=t,
        search_center=center,
        var_map=WEATHERNEXT_VAR_MAP,
    )
    print(
        f"slot {i} ({t:%d %HZ}): p_c {bv.p_c / HPA_TO_PA:.1f} hPa (best track), "
        f"V_m {bv.V_m:.1f} m/s, R_v {bv.R_v_km:.0f} km | bogus storm at "
        f"({s.loc[0]:.2f}, {s.loc[1]:.2f}), {s.mslp_min / HPA_TO_PA:.1f} hPa"
    )
# bv is now the current frame's vortex.

box_check = extract_box(inputs_bogus, bt_center)
snap_bogus0 = snapshot_from_box(
    box_check, valid_time=START_TIME, search_center=bt_center, var_map=WEATHERNEXT_VAR_MAP
)
print(
    f"MSLP min: control {snap_ctrl0.mslp_min / HPA_TO_PA:.1f} hPa -> "
    f"bogus {snap_bogus0.mslp_min / HPA_TO_PA:.1f} hPa"
)
plot_snapshot(
    snap_bogus0,
    box_check,
    var_map=WEATHERNEXT_VAR_MAP,
    title=f"Mawar, ERA5 1° + K&C bogus {START_TIME:%Y-%m-%d %HZ}",
    filename=OUT_DIR / f"snapshot_{BOGUS_LABEL}_t0.png",
)
plt.show()

# Gradient-wind balance after the bogus: K&C builds its wind as the gradient
# wind of its geopotential (reduced in the PBL), so aloft the ratio should
# stay near 1 unless the blend with the analysis breaks it.
plot_snapshot_diagnostics(
    snap_bogus0,
    title=(
        f"Mawar, ERA5 1° + K&C bogus {START_TIME:%Y-%m-%d %HZ}"
        " — gradient-wind balance"
    ),
    filename=OUT_DIR / f"balance_{BOGUS_LABEL}_t0.png",
)
plt.show()

## 6. Load WeatherNext Cyclones Mini

Following `docs/weathernext2/wn2_demo.ipynb`: on GPU the mesh transformer's
attention is switched from TPU splash attention to `triblockdiag_mha`, and the
last predictor wrapper (the training ensemble) is dropped for inference.
`run_forward_jitted` predicts one 6 h step; the first call compiles it.

In [ ]:
transformer_kwargs = config.predictor_kwargs["noisy_function_kwargs"][
    "mesh_model_ctor"
].keywords["transformer_kwargs"]
if jax.default_backend() == "gpu":
    transformer_kwargs["attention_type"] = "triblockdiag_mha"
print("backend:", jax.default_backend(), " attention:", transformer_kwargs["attention_type"])

config_inference = fgn.PredictorConfig(
    task=config.task,
    predictor_constructor=config.predictor_constructor,
    predictor_kwargs=config.predictor_kwargs,
    predictor_wrappers=config.predictor_wrappers[:-1],  # drop the ensemble wrapper
)


@hk.transform
def run_forward(inputs, targets_template, forcings):
    predictor = fgn.construct_predictor(config_inference)
    return predictor(inputs, targets_template=targets_template, forcings=forcings)


@jax.jit
def run_forward_jitted(rng, inputs, targets_template, forcings):
    return run_forward.apply(ckpt.params, rng, inputs, targets_template, forcings)

## 7. Run and track both forecasts

`wn2_states` yields the input dataset, then each forecast step from
`rollout.chunked_prediction_generator` (one 6 h step per chunk, moved to host
memory). Each chunk has one `time` slot, at lead time k × 6 h, so `extract_box`
with `time_idx=-1` reads the right step, and states[0] is the analysis at
`START_TIME`. `track_rollout` consumes them one at a time and keeps only the
storm box.

The model is a functional generative network: each run is one ensemble
member, drawn from the RNG key. Both runs use the same key, so they get the
same noise at every step and differ only through the bogus.

The tracked rollouts are written to `OUT_DIR`. With `REUSE` (section 1),
this cell reads them back instead of running the forecasts. Sections 3–6
still run: they are cheap apart from reading ERA5 from `gs://`, and the
bogus diagnostics come from the inputs. The first `run_forward_jitted` call
compiles the model; with the compilation cache filled, later sessions skip that.

In [ ]:
@contextmanager
def timer(label):
    t0 = time.perf_counter()
    yield
    print(f"  {label}: {time.perf_counter() - t0:8.2f} s", flush=True)


def wn2_states(inp):
    yield inp
    for chunk in rollout.chunked_prediction_generator(
        run_forward_jitted,
        rng=jax.random.PRNGKey(0),
        inputs=inp,
        targets_template=targets_template,
        forcings=forcings,
        num_steps_per_chunk=1,
    ):
        # The generator yields device arrays; copy each step to host, as
        # rollout.chunked_prediction does, so GPU memory is freed per step.
        yield jax.device_get(chunk)


if REUSE:
    ctrl = RolloutSnapshots.from_netcdf(saved_rollout("control"))
    bogus = RolloutSnapshots.from_netcdf(saved_rollout(BOGUS_LABEL))
    print(f"loaded {saved_rollout('control')} ({ctrl.rollout_length} snapshots)")
    print(f"loaded {saved_rollout(BOGUS_LABEL)} ({bogus.rollout_length} snapshots)")
else:
    common = dict(
        cyclone_id=bt.ID,
        start_time=START_TIME,
        initial_center=bt_center,
        step=STEP,
        var_map=WEATHERNEXT_VAR_MAP,
    )
    # The first run's time includes JIT compilation.
    with timer("control forecast + tracking"):
        ctrl = track_rollout(wn2_states(inputs), extract_box, **common)
    with timer("bogus forecast + tracking"):
        bogus = track_rollout(
            wn2_states(inputs_bogus), extract_box, intervened=True, **common
        )

    ctrl.to_netcdf(OUT_DIR / "control")
    bogus.to_netcdf(OUT_DIR / BOGUS_LABEL)

## 8. Compare with the best track

In [ ]:
cols = {
    "model_lead_hours": "lead_h",
    "model_MSLP": "MSLP",
    "model_VMAX": "VMAX",
    "position_error_km": "pos_err_km",
}
df_ctrl = ctrl.to_comparison_dataframe(bt)
df_bogus = bogus.to_comparison_dataframe(bt)
table = pd.concat(
    {
        "control": df_ctrl[list(cols)].rename(columns=cols),
        "bogus": df_bogus[list(cols)].rename(columns=cols).drop(columns="lead_h"),
        "best track": df_ctrl[["bt_MSLP", "bt_VMAX"]].rename(
            columns={"bt_MSLP": "MSLP", "bt_VMAX": "VMAX"}
        ),
    },
    axis=1,
)
table.round(1)

In [ ]:
plot_best_track_comparison(ctrl, bogus, bt, title=f"WN Cyclones Mini <{SPLIT}, Mawar from {START_TIME:%Y-%m-%d %HZ}")

In [ ]:
plot_rollout_comparison(
    ctrl,
    bogus,
    OUT_DIR / f"comparison_wn2_mawar{OMEGA_TAG}.pdf",
    best_track=bt,
    labels=(
        "WN Cyclones Mini control",
        "WN Cyclones Mini + K&C bogus" + (" + omega" if BLEND_OMEGA else ""),
    ),
)
plot_rollout_snapshots(ctrl, OUT_DIR / "rollout_wn2_control.pdf", best_track=bt)
plot_rollout_snapshots(bogus, OUT_DIR / f"rollout_wn2_{BOGUS_LABEL}.pdf", best_track=bt)
print(sorted(p.name for p in OUT_DIR.glob("*.pdf")))